# 04 — Baseline models

**Objective.** Train TF-IDF + Naive Bayes / Logistic Regression / Linear SVM with leakage-aware splits, and compare against dummy baselines.

In [1]:
import sys, json, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (9, 4)
from ml.config import PROCESSED_DATASET, ARTIFACTS_DIR, METADATA_DIR, RAW_DIR
df = pd.read_csv(PROCESSED_DATASET)
print(f'{len(df):,} processed fact-checks')

4,137 processed fact-checks


In [2]:
from ml.training.features import prepare_training_frame, build_vectorizer, mask_verdict_cues
from ml.training.splits import group_split
data = prepare_training_frame(df)
train, val, test = group_split(data)
print(len(train), len(val), len(test)); train['label'].value_counts(normalize=True).round(3)

2591 519 519


label
FALSE         0.421
MISLEADING    0.372
OUTDATED      0.206
Name: proportion, dtype: float64

## Leakage check
Headlines contain the verdict ("… is a deepfake"). For title-only rows we mask the cue words before training:

In [3]:
for t in df[df.claim_source=='title'].fact_check_title.head(4): print(t, '\n  ->', mask_verdict_cues(t))

नेहा बोरा ने भारत को 'बांग्लादेश' बनाने की बात नहीं की थी; वायरल वीडियो में अहम संदर्भ छूट गया 
  -> नेहा बोरा ने भारत को 'बांग्लादेश' बनाने की बात नहीं की थी; वायरल वीडियो में अहम
Media misreport: Akhilesh Yadav put on a red turban at Lucknow press meet, not saffron 
  -> Media misreport: Akhilesh Yadav put on a red turban at Lucknow press meet affron
एडिटेड वीडियो: रिटायर्ड लेफ्टिनेंट जनरल मनोज कुमार कटियार ने अरुणाचल प्रदेश को लेकर चेतावनी नहीं दी थी 
  -> वीडियो: रिटायर्ड लेफ्टिनेंट जनरल मनोज कुमार कटियार ने अरुणाचल प्रदेश को लेकर चेतावनी नहीं दी थी
चीन की गाड़ियों का भारत में घुसने का दावा भ्रामक, नाथू ला सीमा व्यापार से जुड़ा है वायरल वीडियो 
  -> चीन की गाड़ियों का भारत में घुसने का दावा , नाथू ला सीमा व्यापार से जुड़ा है वायरल वीडियो


In [4]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.dummy import DummyClassifier
from sklearn.metrics import f1_score
res = {}
for name, clf in {'dummy': DummyClassifier(strategy='stratified', random_state=42), 'nb': MultinomialNB(alpha=0.1),
                  'logreg': LogisticRegression(C=4, max_iter=3000, class_weight='balanced')}.items():
    p = Pipeline([('features', build_vectorizer()), ('clf', clf)]).fit(train.text, train.label)
    res[name] = f1_score(val.label, p.predict(val.text), average='macro')
pd.Series(res).round(3).to_frame('validation macro-F1')

,validation macro-F1
dummy,0.306
nb,0.542
logreg,0.584


## The full, saved comparison (from `ml/training/train_classical.py`)

In [5]:
meta = json.loads((ARTIFACTS_DIR / 'classifier_metadata.json').read_text())
pd.DataFrame({k: {'val_macro_f1': v['validation']['f1_macro'], 'test_macro_f1': v['test']['f1_macro'], 'test_acc': v['test']['accuracy'], 'test_ece': v['test']['calibration']['ece']} for k, v in meta['results'].items()}).T

,val_macro_f1,test_macro_f1,test_acc,test_ece
dummy_most_frequent,0.1978,0.1978,0.4220,0.5780
dummy_stratified,0.3063,0.3047,0.3391,0.6609
multinomial_nb,0.5425,0.5050,0.5337,0.3841
complement_nb,0.5375,0.5031,0.5414,0.3167
logistic_regression,0.5841,0.5270,0.5549,0.1254
linear_svm,0.5454,0.5065,0.5742,0.0616


## Observations
* All real models beat the stratified baseline (~0.31 macro-F1) by a wide margin, but none exceeds ~0.6.
* Logistic Regression is best on validation and best calibrated after the SVM.

**Conclusion.** Deploy Logistic Regression: competitive and exactly explainable.